# AlphaGenome Fine-Tuning Notebook

This notebook is designed for practical, editable fine-tuning runs.

Workflow:
1. Set paths and training options
2. (Optional) Download missing files
3. Validate inputs
4. Build heads and intervals
5. Initialize model
6. Train and checkpoint


## 0) Imports

In [3]:
from __future__ import annotations

from pathlib import Path
import urllib.request

from alphagenome_ft import create_model_with_heads, load_checkpoint
from alphagenome_ft.finetune.config import (
    build_head_specs,
    build_head_specs_from_config,
    validate_heads,
)
from alphagenome_ft.finetune.data import (
    BigWigDataModule,
    get_fold_intervals,
    load_intervals_from_dataframe,
)
from alphagenome_ft.finetune.train import register_predefined_heads, train


## 1) User Config

Edit these values first. Keep paths relative to your project root if possible.

In [4]:
# Core input files
PROJECT_DIR = Path("./")  # Set this to the root of the project
PROJECT_DIR = Path("../data/")
FASTA_PATH = PROJECT_DIR / "hg38.fa"

# Target config mode
TARGETS_CONFIG_PATH = PROJECT_DIR / "targets_rna_demo.yaml"  # Used when USE_INLINE_TARGETS=False

# INLINE_HEADS_CONFIG = {
#     "heads": [
#         {
#             "id": "example_rna_head",
#             "source": "predefined",
#             "kind": "rna_seq",
#             "targets": [
#                 {"bigwig": str(PROJECT_DIR / "data/example_rna_rep1_chr1.bw"), "label": "rna_1"},
#                 {"bigwig": str(PROJECT_DIR / "data/example_rna_rep2_chr1.bw"), "label": "rna_2"},
#             ],
#         }
#     ]
# }

# Data options
FOLD = "1"  # 0, 1, 2, 3
WINDOW_SIZE = 1_048_576  # 1 Mbp window

# Model options
ORGANISM = "HOMO_SAPIENS"
MODEL_VERSION = "fold_1"  # fold_0, fold_1, fold_2, fold_3, all_folds
HEADS_ONLY = True

# TODO: Rename
CHECKPOINT_PATH = None  # Path(...) to initialize from a local base checkpoint

# Training options
BATCH_SIZE = 1
NUM_EPOCHS = 10
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
SEED = 42
SHUFFLE = True
DROP_LAST = False
MAX_TRAIN_STEPS = None
VERBOSE = True

# Checkpointing / early stopping
CHECKPOINT_DIR = Path("checkpoints/finetune_demo_notebook")
BEST_METRIC = "valid_loss"
BEST_METRIC_MODE = "min"
EARLY_STOPPING_PATIENCE = 5
EARLY_STOPPING_MIN_DELTA = 0.0


## 2) Optional FASTA Download

Default is disabled. This helper only applies to `hg38.fa` in this notebook.


In [5]:
ENABLE_DOWNLOAD = False
FASTA_DOWNLOAD_URL = "https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.fa.gz"

if ENABLE_DOWNLOAD:
    FASTA_PATH.parent.mkdir(parents=True, exist_ok=True)
    if FASTA_PATH.exists():
        print(f"Skip existing: {FASTA_PATH}")
    else:
        print(f"Downloading: {FASTA_DOWNLOAD_URL} -> {FASTA_PATH}")
        urllib.request.urlretrieve(FASTA_DOWNLOAD_URL, FASTA_PATH)
else:
    print("Download step is disabled.")


Download step is disabled.


## 3) Validate Inputs

In [6]:
for path in [FASTA_PATH, TARGETS_CONFIG_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

if CHECKPOINT_DIR is not None:
    CHECKPOINT_DIR = Path(CHECKPOINT_DIR).resolve()
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if CHECKPOINT_PATH is not None:
    CHECKPOINT_PATH = Path(CHECKPOINT_PATH).resolve()

print("Input validation passed.")

Input validation passed.


## 4) Build Head Specs and Fold Intervals


In [7]:
head_specs = build_head_specs(TARGETS_CONFIG_PATH, organism=ORGANISM)

validate_heads(head_specs)
register_predefined_heads(head_specs)

# Create dataframe of intervals for the specified fold and window size
intervals_df = get_fold_intervals(
    fold=FOLD,
    window_size=WINDOW_SIZE,
    organism=ORGANISM,
)

# Create a dict of {split: alphagenome.data.genome.Interval} for data module
intervals = load_intervals_from_dataframe(intervals_df)

print("Heads:", [spec.head_id for spec in head_specs])
for split in ("train", "valid", "test"):
    print(f"{split}: {len(intervals.get(split, []))} intervals")

Filtered out 1265 intervals from valid/test sets due to training overlap.
Heads: ['my_rna_head']
train: 42258 intervals
valid: 5877 intervals
test: 6097 intervals


## 5) Initialize or Resume Model

In [8]:
model = create_model_with_heads(
    MODEL_VERSION,
    heads=[spec.head_id for spec in head_specs],
    detach_backbone=HEADS_ONLY,
    checkpoint_path=CHECKPOINT_PATH,
)

print("Model ready.")

Loading pretrained AlphaGenome model...

/grid/koo/home/nagai/projects/alphagenome_collab/.venv/lib/python3.12/site-packages/pyfaidx/__init__.py:589: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)
/grid/koo/home/nagai/projects/alphagenome_collab/.venv/lib/python3.12/site-packages/pyfaidx/__init__.py:589: UserWarning: for fsspec: HTTPFileSystem assuming index is current
  warnings.warn("for fsspec: %s assuming index is current" % type(self._fs).__name__)


✓ Pretrained model loaded
Initializing heads: ['my_rna_head']
Initializing parameters... (seq_len=16384)
✓ Head parameters initialized
Merging pretrained backbone with heads...
✓ Parameters merged
✓ Model created successfully
  Total parameters: 450,471,061
  Heads: ['my_rna_head']
Model ready.


## 6) Build Data Module

In [9]:
data_module = BigWigDataModule(
    intervals=intervals,
    fasta_path=FASTA_PATH,
    head_specs=head_specs,
    batch_size=BATCH_SIZE,
    shuffle=SHUFFLE,
    drop_last=DROP_LAST,
)

print("Data module ready.")

Filtered out 49863 intervals to match BigWig chromosome coverage (removed chromosomes: chr10, chr11, chr12, chr13, chr14, chr15, chr16, chr17, chr18, chr19, chr2, chr20, chr21, chr22, chr3, chr4, chr5, chr6, chr7, chr8, chr9, chrX; removed intervals: train=38327, valid=5676, test=5860).
Data module ready.


## 7) (Optional) Smoke Test Settings

Use this to test your setup quickly before a long run.

In [10]:
SMOKE_TEST = True
if SMOKE_TEST:
    NUM_EPOCHS = 1
    MAX_TRAIN_STEPS = 20
    print("Smoke test enabled.")
else:
    print("Smoke test disabled.")


Smoke test enabled.


## 8) Train

In [11]:
train(
    model,
    data_module,
    head_specs,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_epochs=NUM_EPOCHS,
    seed=SEED,
    max_train_steps=MAX_TRAIN_STEPS,
    heads_only=HEADS_ONLY,
    checkpoint_dir=CHECKPOINT_DIR,
    organism=ORGANISM,
    best_metric=BEST_METRIC,
    best_metric_mode=BEST_METRIC_MODE,
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
    verbose=VERBOSE,
)

JIT-compiling step functions (first call will be slow)...
Train plan: 3931 examples | 3931 step(s)/epoch | 1 epoch(s) | total step(s) 20

Epoch 1/1
  step 0020/3931 | loss 5.55581
  Train loss: 3.9604
  Validation metrics: my_rna_head=7.6933
  Metric improved (valid_loss = 7.6933)  -- saving best checkpoint
✓ Checkpoint saved to /grid/koo/home/nagai/projects/alphagenome_collab/alphagenome_ft/notebooks/checkpoints/finetune_demo_notebook/best
  Saved: Heads only ['my_rna_head']
✓ Checkpoint saved to /grid/koo/home/nagai/projects/alphagenome_collab/alphagenome_ft/notebooks/checkpoints/finetune_demo_notebook/last
  Saved: Heads only ['my_rna_head']
  Reached requested training steps: 20/20

Training complete!
